# TakeMeter — Fine-Tuning Starter Notebook
### AI201 · Project 3

This notebook walks you through fine-tuning a text classifier on your annotated dataset and comparing it to a zero-shot baseline.

**What this notebook does for you (infrastructure):**
- Tokenizes your dataset and prepares it for training
- Runs the fine-tuning pipeline with DistilBERT
- Computes evaluation metrics and generates a confusion matrix
- Runs the Groq baseline and compares both models

**What you do (the actual work):**
- Collect and annotate your 200+ examples (done before opening this notebook)
- Define your label map and upload your CSV
- Write your Groq classification prompt using your label definitions
- Analyze the output and write your evaluation report

---
**Before you start:** Make sure you are using a T4 GPU runtime.  
Go to **Runtime → Change runtime type → T4 GPU**, then click Save.


---
## Completed Submission Note

This is the completed version of the official AI201 Project 3 starter notebook for **TakeMeter**, a classifier for r/sports comments. It uses the submitted dataset at `data/takemeter_labeled.csv` (252 labeled comments), the project taxonomy (`reaction`, `hot_take`, `analysis`), the same deterministic 70/15/15 split as `src/dataset.py`, and the same result framing as `README.md` and `evaluation_results.json`.

Current local artifact results, already saved in the repo:

| model | accuracy | macro-F1 |
|---|---:|---:|
| Fine-tuned DistilBERT | 0.632 | 0.479 |
| Zero-shot Groq baseline (`llama-3.3-70b-versatile`) | 0.921 | 0.901 |

The notebook remains runnable end-to-end in Colab. Re-running the Groq baseline requires a `GROQ_API_KEY` secret.


In [ ]:
# Install the dependencies this notebook needs in Colab.
!pip -q install "transformers>=4.46" "datasets>=2.20" "accelerate>=1.1.0" scikit-learn matplotlib groq pandas python-dotenv
print("Dependencies ready")


In [ ]:
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
import matplotlib.pyplot as plt

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

print("Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


---
## Section 1: Load Your Dataset

Upload your labeled CSV and define your label map.  
Your CSV must have at least two columns: `text` (the post/comment) and `label` (your string label).


In [ ]:
# TakeMeter label map.
# Order is fixed to match src/labels.py, the saved model, and evaluation artifacts.
LABEL_MAP = {
    "reaction": 0,
    "hot_take": 1,
    "analysis": 2,
}

ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)
print(f"Labels: {LABEL_MAP}")
print(f"Number of labels: {NUM_LABELS}")


In [ ]:
# Use the checked-in labeled CSV when this notebook is run from the repo root.
# In Colab, upload data/takemeter_labeled.csv if the file is not already present.
DEFAULT_CSV_PATHS = [
    Path("data/takemeter_labeled.csv"),
    Path("../data/takemeter_labeled.csv"),
    Path("/content/takemeter_labeled.csv"),
]

CSV_PATH = next((str(path) for path in DEFAULT_CSV_PATHS if path.exists()), None)

if CSV_PATH is None:
    try:
        from google.colab import files
        print("Select data/takemeter_labeled.csv...")
        uploaded = files.upload()
        CSV_PATH = list(uploaded.keys())[0]
    except Exception as exc:
        raise FileNotFoundError(
            "Could not find data/takemeter_labeled.csv. Run from the repo root "
            "or upload the CSV in Colab."
        ) from exc

print(f"Using CSV: {CSV_PATH}")


In [ ]:
# Load and validate the submitted dataset.
df = pd.read_csv(CSV_PATH)
required = {"text", "label"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {sorted(missing)}")

keep_cols = ["text", "label"] + (["notes"] if "notes" in df.columns else [])
df = df[keep_cols].copy()
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip()
df = df[df["text"].ne("")].reset_index(drop=True)

# Leakage guard: drop near-duplicate texts before splitting.
before = len(df)
norm = df["text"].str.lower().str.replace(r"\s+", " ", regex=True)
df = df[~norm.duplicated()].reset_index(drop=True)
if before != len(df):
    print(f"Dropped {before - len(df)} duplicate-text rows")

print(f"Columns: {df.columns.tolist()}")
print(f"Total examples: {len(df)}")
print()
print("Label distribution:")
counts = df["label"].value_counts().reindex(LABEL_MAP.keys(), fill_value=0)
print(counts)

unknown = set(df["label"].unique()) - set(LABEL_MAP.keys())
if unknown:
    raise ValueError(f"Labels in CSV not found in LABEL_MAP: {sorted(unknown)}")

if len(df) < 200:
    raise ValueError(f"Need at least 200 labeled examples; found {len(df)}")

shares = counts / len(df)
if (shares < 0.20).any() or (shares > 0.70).any():
    raise ValueError(f"Class balance outside 20%-70% requirement: {shares.round(3).to_dict()}")

print("All labels match LABEL_MAP and class balance passes the assignment gate")

df["label_id"] = df["label"].map(LABEL_MAP).astype(int)


---
## Section 2: Prepare Data for Training

Splits your dataset into train / validation / test sets and tokenizes the text.


In [ ]:
# Train / val / test split: exact same deterministic 70/15/15 logic as src/dataset.py.
SEED = 42

def stratified_split_by_label(frame, seed=SEED):
    train_parts, val_parts, test_parts = [], [], []
    for label in LABEL_MAP.keys():
        sub = frame[frame["label"] == label].sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n = len(sub)
        if n < 4:
            raise ValueError(f"class '{label}' has only {n} examples; collect more before splitting")
        n_test = max(1, round(n * 0.15))
        n_val = max(1, round(n * 0.15))
        test_parts.append(sub.iloc[:n_test])
        val_parts.append(sub.iloc[n_test:n_test + n_val])
        train_parts.append(sub.iloc[n_test + n_val:])

    train = pd.concat(train_parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    val = pd.concat(val_parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    test = pd.concat(test_parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

    s_train, s_val, s_test = set(train.text), set(val.text), set(test.text)
    assert not (s_train & s_val), "leakage: train/val text overlap"
    assert not (s_train & s_test), "leakage: train/test text overlap"
    assert not (s_val & s_test), "leakage: val/test text overlap"
    return train, val, test

train_df, val_df, test_df = stratified_split_by_label(df)

print(f"Train: {len(train_df)} examples")
print(f"Validation: {len(val_df)} examples")
print(f"Test: {len(test_df)} examples")
print()
print("Train label distribution:")
print(train_df["label"].value_counts().reindex(LABEL_MAP.keys(), fill_value=0))
print()
print("Test label distribution:")
print(test_df["label"].value_counts().reindex(LABEL_MAP.keys(), fill_value=0))

Path("artifacts").mkdir(exist_ok=True)
train_df.to_csv("artifacts/train.csv", index=False)
val_df.to_csv("artifacts/val.csv", index=False)
test_df.to_csv("artifacts/test.csv", index=False)
print("Saved artifacts/train.csv, artifacts/val.csv, artifacts/test.csv")


In [ ]:
# Load tokenizer and tokenize all splits.
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LEN)

def make_dataset(df_split):
    ds = Dataset.from_dict({
        "text": df_split["text"].tolist(),
        "labels": df_split["label_id"].astype(int).tolist(),
    })
    return ds.map(tokenize, batched=True)

train_dataset = make_dataset(train_df)
val_dataset = make_dataset(val_df)
test_dataset = make_dataset(test_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenization complete")
print(f"Sample keys: {list(train_dataset[0].keys())}")


---
## Section 3: Fine-Tune the Model

Loads `distilbert-base-uncased` with a classification head and fine-tunes it on the TakeMeter training data.

**Hyperparameter decision:** the course default is 3 epochs. This submitted run uses 5 epochs with early stopping on validation macro-F1, matching `src/finetune.py` and the README. With only 176 training examples and a subtle `hot_take` boundary, the extra epochs let the model train longer while still keeping the best validation checkpoint instead of blindly using the last epoch.


In [ ]:
# Load DistilBERT with a classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP,
)
print(f"✅ Model loaded: {MODEL_NAME}")
print(f"Output labels: {NUM_LABELS}")


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
    }


In [ ]:
training_args = TrainingArguments(
    output_dir="./takemeter-model",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete")


---
## Section 4: Evaluate Fine-Tuned Model on Test Set

Runs inference on your locked test set and generates metrics and a confusion matrix.  
These numbers go directly into your evaluation report.


In [ ]:
# Run inference on the locked test set.
print("Running inference on test set...")
ft_output = trainer.predict(test_dataset)
ft_pred_ids = np.argmax(ft_output.predictions, axis=-1)
ft_true_ids = ft_output.label_ids

ft_probs = torch.nn.functional.softmax(
    torch.tensor(ft_output.predictions), dim=-1
).numpy()

ft_accuracy = accuracy_score(ft_true_ids, ft_pred_ids)
ft_macro_f1 = f1_score(ft_true_ids, ft_pred_ids, average="macro")
print(f"Fine-tuned model accuracy: {ft_accuracy:.3f}")
print(f"Fine-tuned model macro-F1: {ft_macro_f1:.3f}")

label_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
print("\nPer-class metrics (fine-tuned model):")
print(classification_report(ft_true_ids, ft_pred_ids, target_names=label_names, zero_division=0))


In [ ]:
# Confusion matrix.
cm = confusion_matrix(ft_true_ids, ft_pred_ids)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Fine-Tuned Model - Confusion Matrix (Test Set)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("Saved confusion_matrix.png")


In [ ]:
# Print wrong predictions for your error analysis
# Review these carefully — pick 3 to analyze in depth in your README.

wrong_idx = np.where(ft_pred_ids != ft_true_ids)[0]
print(f"Wrong predictions: {len(wrong_idx)} / {len(ft_true_ids)}\n")

for i, idx in enumerate(wrong_idx[:15]):
    text = test_df.iloc[idx]["text"]
    true_label = ID_TO_LABEL[ft_true_ids[idx]]
    pred_label = ID_TO_LABEL[ft_pred_ids[idx]]
    confidence = ft_probs[idx][ft_pred_ids[idx]]
    print(f"--- #{i+1} ---")
    print(f"Text:      {text[:200]}{'...' if len(text) > 200 else ''}")
    print(f"True:      {true_label}")
    print(f"Predicted: {pred_label}  (confidence: {confidence:.2f})")
    print()


---
## Section 5: Baseline Classifier (Groq)

Runs the zero-shot baseline on the same locked test split.

This section uses the rubric-required `llama-3.3-70b-versatile` baseline on the same locked test split. It is a different model from the `llama-4-scout-17b` annotation helper, so the comparison is not circular.


In [ ]:
from groq import Groq

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")

assert GROQ_API_KEY, (
    "GROQ_API_KEY not set. In Colab, add it in Secrets and enable notebook access. "
    "Locally, put it in .env or export it in your shell."
)

BASELINE_MODEL = os.getenv("TAKEMETER_BASELINE_MODEL", "llama-3.3-70b-versatile")
client = Groq(api_key=GROQ_API_KEY)
print(f"Groq client initialized with baseline model: {BASELINE_MODEL}")


In [ ]:
SYSTEM_PROMPT = """
You are classifying one Reddit r/sports comment by the kind of discourse it is.
Assign exactly one label.

Decision tree:
Q1. Does the comment make a debatable sports claim, opinion, prediction, or ranking?
- If no, label it reaction.
Q2. If yes, is the claim backed by a real explanatory or causal chain, or multiple pieces of evidence?
- If no, label it hot_take.
- If yes, label it analysis.

Labels:
reaction: Emotional, in-the-moment expression that makes no substantive sports claim: hype, disbelief, humor, despair, or pure fandom.
Example: "WHAT A GOAL ARE YOU KIDDING ME"

hot_take: Strongly-stated, debatable sports opinion, claim, prediction, or ranking with little or no supporting reasoning. A single stat dropped after the claim is garnish, not an argument.
Example: "He's washed, time to admit it"

analysis: Reasoned, evidence-based sports argument that explains why through a causal chain or multiple pieces of evidence such as tactics, stats, matchups, film, or history.
Example: "They keep blitzing on third down, so until the back picks it up the play-action never develops and the offense stalls."

Tiebreakers:
- An emotional outburst that also asserts a debatable sports claim is hot_take.
- A claim with only one stat tacked on is hot_take.
- Sarcasm and jokes are labeled by the proposition they imply.
- A pure meme or celebration with no claim is reaction.

Respond with ONLY one exact label:
reaction
hot_take
analysis
"""

print("Prompt length:", len(SYSTEM_PROMPT), "characters")


In [ ]:
def parse_groq_label(raw_text):
    raw = raw_text.strip().lower()
    first_line = raw.splitlines()[0].strip().strip('`"\' .:-') if raw else ""
    normalized = first_line.replace("hot take", "hot_take").replace("-", "_").replace(" ", "_")
    if normalized in LABEL_MAP:
        return normalized
    for label in sorted(LABEL_MAP, key=len, reverse=True):
        if re.search(rf"\b{re.escape(label)}\b", raw):
            return label
    if re.search(r"\bhot\s+take\b", raw):
        return "hot_take"
    return None


def classify_with_groq(text):
    """Classify a single comment. Returns a label string or None if unparseable."""
    try:
        response = client.chat.completions.create(
            model=BASELINE_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Classify this r/sports comment:\n\n{text}"},
            ],
            temperature=0,
            max_tokens=20,
        )
        raw = response.choices[0].message.content
        return parse_groq_label(raw)
    except Exception as e:
        print(f"API error: {e}")
        return None


print(f"Running baseline on {len(test_df)} examples...")
print("This may take a few minutes; there is a short delay between requests.")

baseline_preds = []
for i, (_, row) in enumerate(test_df.iterrows()):
    pred = classify_with_groq(row["text"])
    baseline_preds.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(test_df)} complete")
    time.sleep(0.1)

none_count = baseline_preds.count(None)
print(f"Unparseable responses: {none_count}/{len(test_df)}")
if none_count > 0:
    print("Review prompt/model output before using baseline metrics.")


In [ ]:
# Baseline metrics (exclude unparseable responses).
valid = [(p, t) for p, t in zip(baseline_preds, test_df["label_id"]) if p is not None]
if not valid:
    raise ValueError("No parseable baseline responses; cannot compute metrics")

bl_pred_ids = [LABEL_MAP[p] for p, _ in valid]
bl_true_ids = [int(t) for _, t in valid]

bl_accuracy = accuracy_score(bl_true_ids, bl_pred_ids)
bl_macro_f1 = f1_score(bl_true_ids, bl_pred_ids, average="macro")
print(f"Baseline accuracy: {bl_accuracy:.3f} (evaluated on {len(valid)}/{len(test_df)} parseable responses)")
print(f"Baseline macro-F1: {bl_macro_f1:.3f}")
print()
label_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
print("Per-class metrics (baseline):")
print(classification_report(bl_true_ids, bl_pred_ids, target_names=label_names, zero_division=0))


---
## Section 6: Compare Results and Export

Side-by-side comparison of both models.  
Download the output files and commit them to your GitHub repo.


In [ ]:
print("=" * 64)
print("RESULTS COMPARISON")
print("=" * 64)
print(f"{'Model':<35} {'Accuracy':>10} {'Macro-F1':>10}")
print("-" * 58)
print(f"{f'Zero-shot baseline ({BASELINE_MODEL})':<35} {bl_accuracy:>10.3f} {bl_macro_f1:>10.3f}")
print(f"{'Fine-tuned DistilBERT':<35} {ft_accuracy:>10.3f} {ft_macro_f1:>10.3f}")
print("-" * 58)
macro_delta = ft_macro_f1 - bl_macro_f1
acc_delta = ft_accuracy - bl_accuracy
print(f"Fine-tuned accuracy delta: {acc_delta:+.3f}")
print(f"Fine-tuned macro-F1 delta: {macro_delta:+.3f}")
print()
print("Use these numbers in the README evaluation report.")


In [ ]:
# Save results JSON for the README and submission artifacts.
results = {
    "fine_tuned": {
        "accuracy": round(float(ft_accuracy), 4),
        "macro_f1": round(float(ft_macro_f1), 4),
        "n": int(len(test_df)),
    },
    "baseline_zero_shot": {
        "model": BASELINE_MODEL,
        "accuracy": round(float(bl_accuracy), 4),
        "macro_f1": round(float(bl_macro_f1), 4),
        "n": int(len(valid)),
        "unparseable_count": int(none_count),
    },
    "delta_macro_f1_ft_minus_baseline": round(float(ft_macro_f1 - bl_macro_f1), 4),
    "test_set_size": int(len(test_df)),
    "label_map": LABEL_MAP,
    "model": MODEL_NAME,
}
with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Files ready:")
print("   evaluation_results.json")
print("   confusion_matrix.png")
print("   artifacts/train.csv, artifacts/val.csv, artifacts/test.csv")


---
## Submitted Result Snapshot

The committed repo already includes the completed artifacts from this run:

- `data/takemeter_labeled.csv`: 252 labeled r/sports comments.
- `artifacts/test.csv`: locked 38-row held-out test split used by both models.
- `evaluation_results.json`: fine-tuned and baseline metrics.
- `confusion_matrix.png`: fine-tuned confusion matrix.
- `README.md`: evaluation report, error analysis, sample classifications, spec reflection, and AI usage disclosure.

If this notebook is rerun, the exact numeric results may vary slightly because fine-tuning is stochastic. The submission report uses the saved artifacts above.
